# IT Asset Management (ITAM) Database – Starter Analysis Notebook

**Author:** Randall James  
**Purpose:** Demonstrate Python + PostgreSQL integration for data analysis workflows. This notebook connects to the ITAM database, runs several production-style analytical queries, and produces pandas DataFrames + visualizations.

This is intended as a **portfolio starter** for IT Data Analyst / BI Analyst / Data Coordinator roles. It shows:
- Clean database connection patterns (SQLAlchemy + pandas)
- Reusable query execution helper
- Real analytical questions (compliance, cost analysis, utilization)
- Basic visualization with seaborn/matplotlib

**Prerequisites:**
1. Run `docker compose up -d` in the project root
2. Install requirements: `pip install -r analysis/requirements.txt`
3. Make sure the database has been seeded (`sql/02_seed_data.sql`)

In [ ]:
# =============================================================================
# Imports & Configuration
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully.")

In [ ]:
# =============================================================================
# Database Connection
# =============================================================================
# Update these values if you changed the docker-compose defaults
DB_USER = "postgres"
DB_PASSWORD = "supersecurepassword123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "itam_db"

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(connection_string)
    # Test connection
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("Connected to PostgreSQL:", result.scalar())
except Exception as e:
    print("Connection failed:", e)
    print("Make sure docker compose up -d has been run and the DB is seeded.")

In [ ]:
# =============================================================================
# Helper Function: Run SQL → pandas DataFrame
# =============================================================================
def run_query(sql: str, params: dict = None) -> pd.DataFrame:
    """Execute SQL and return results as a pandas DataFrame."""
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn, params=params)
    return df

print("Helper function ready.")

## 1. Warranty Expiring Soon Report (Compliance Alert)

Business question: Which assets have warranties expiring in the next 90 days? This is a common monthly compliance report for IT teams.

In [ ]:
warranty_sql = """
WITH expiring_warranty AS (
    SELECT 
        a.asset_tag,
        a.serial_number,
        at.name AS asset_type,
        a.manufacturer || ' ' || a.model AS model,
        a.purchase_date,
        a.warranty_end_date,
        a.purchase_cost,
        e.first_name || ' ' || e.last_name AS assigned_to,
        d.name AS department,
        (a.warranty_end_date - CURRENT_DATE) AS days_until_expiry
    FROM assets a
    JOIN asset_types at ON a.asset_type_id = at.asset_type_id
    LEFT JOIN employees e ON a.assigned_to_employee_id = e.employee_id
    LEFT JOIN departments d ON e.department_id = d.department_id
    WHERE a.warranty_end_date IS NOT NULL
      AND a.warranty_end_date BETWEEN CURRENT_DATE AND CURRENT_DATE + INTERVAL '90 days'
      AND a.status IN ('In Use', 'In Storage')
)
SELECT 
    *,
    CASE 
        WHEN days_until_expiry <= 30 THEN 'CRITICAL'
        WHEN days_until_expiry <= 60 THEN 'HIGH'
        ELSE 'MEDIUM'
    END AS urgency
FROM expiring_warranty
ORDER BY days_until_expiry ASC;
"""

df_warranty = run_query(warranty_sql)
print(f"Found {len(df_warranty)} assets with warranties expiring soon.")
df_warranty.head(10)

## 2. Asset Value & Estimated Book Value by Department

Business question: What is the total original cost and estimated current book value of IT assets assigned to each department? (Simple straight-line depreciation example)

In [ ]:
asset_value_sql = """
SELECT 
    d.name AS department,
    COUNT(a.asset_id) AS total_assets,
    COUNT(*) FILTER (WHERE a.status = 'In Use') AS assets_in_use,
    ROUND(SUM(a.purchase_cost), 2) AS total_original_cost,
    ROUND(AVG(a.purchase_cost), 2) AS avg_asset_cost,
    -- Simple straight-line depreciation (assume 4-year life)
    ROUND(SUM(
        a.purchase_cost * GREATEST(0, 1 - (EXTRACT(YEAR FROM AGE(CURRENT_DATE, a.purchase_date)) / 4.0))
    ), 2) AS estimated_current_book_value
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id AND e.status = 'Active'
LEFT JOIN assets a ON e.employee_id = a.assigned_to_employee_id 
                   AND a.status IN ('In Use', 'In Storage')
GROUP BY d.department_id, d.name
ORDER BY total_original_cost DESC;
"""

df_asset_value = run_query(asset_value_sql)
df_asset_value

In [ ]:
# Visualization: Total Original Cost by Department
plt.figure(figsize=(11, 6))
sns.barplot(
    data=df_asset_value.sort_values('total_original_cost', ascending=False),
    x='total_original_cost',
    y='department',
    palette='viridis'
)
plt.title('Total IT Asset Original Cost by Department', fontsize=14, pad=15)
plt.xlabel('Total Original Cost (USD)')
plt.ylabel('Department')
plt.tight_layout()
plt.show()

## 3. License Utilization Dashboard

Business question: What is the current seat utilization percentage for each software license? Which licenses are expiring soon or already expired?

In [ ]:
license_sql = """
WITH current_allocations AS (
    SELECT 
        license_id,
        SUM(seats_allocated) AS currently_allocated
    FROM license_allocations
    WHERE returned_date IS NULL
    GROUP BY license_id
)
SELECT 
    sp.name AS software,
    sp.publisher,
    sl.total_seats,
    COALESCE(ca.currently_allocated, 0) AS allocated,
    ROUND(100.0 * COALESCE(ca.currently_allocated, 0) / NULLIF(sl.total_seats, 0), 1) AS utilization_pct,
    sl.expiration_date,
    CASE 
        WHEN sl.expiration_date < CURRENT_DATE THEN 'EXPIRED'
        WHEN sl.expiration_date <= CURRENT_DATE + INTERVAL '90 days' THEN 'EXPIRING SOON'
        ELSE 'ACTIVE'
    END AS renewal_status
FROM software_licenses sl
JOIN software_products sp ON sl.product_id = sp.product_id
LEFT JOIN current_allocations ca ON sl.license_id = ca.license_id
ORDER BY utilization_pct DESC NULLS LAST;
"""

df_licenses = run_query(license_sql)
df_licenses

## 4. Top Employees by Asset Value (Window Function Example)

Business question: Which employees have the highest total asset value assigned? Show ranking within their department.

In [ ]:
top_employees_sql = """
WITH employee_asset_cost AS (
    SELECT 
        e.employee_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        e.job_title,
        d.name AS department,
        COUNT(a.asset_id) AS asset_count,
        COALESCE(SUM(a.purchase_cost), 0) AS total_asset_value
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    LEFT JOIN assets a ON e.employee_id = a.assigned_to_employee_id 
                       AND a.status = 'In Use'
    WHERE e.status = 'Active'
    GROUP BY e.employee_id, e.first_name, e.last_name, e.job_title, d.name
)
SELECT 
    employee_name,
    job_title,
    department,
    asset_count,
    total_asset_value,
    RANK() OVER (ORDER BY total_asset_value DESC) AS company_rank,
    RANK() OVER (PARTITION BY department ORDER BY total_asset_value DESC) AS dept_rank
FROM employee_asset_cost
ORDER BY total_asset_value DESC
LIMIT 15;
"""

df_top_employees = run_query(top_employees_sql)
df_top_employees

## Next Steps & Extensions

This notebook demonstrates core patterns. Here are natural extensions you can add:

1. **Parameterized queries** – Use SQLAlchemy `bindparams` or `text()` with parameters for dynamic date ranges.
2. **Scheduled reporting** – Turn key queries into functions and schedule via Airflow, Prefect, or cron + email.
3. **More visualizations** – Add time-series line charts for maintenance spend trends or license utilization over time.
4. **Export to Excel** – Use `df.to_excel()` with openpyxl or xlsxwriter for stakeholder reports.
5. **Dashboarding** – Feed these DataFrames into Streamlit, Dash, or Metabase.

**Pro tip for interviews:** Be ready to explain *why* you chose SQLAlchemy + pandas over raw psycopg2, and how you would productionize these queries (views, materialized views, dbt models, etc.).

In [ ]:
# Optional: Close engine connection pool when done
# engine.dispose()
print("Notebook execution complete. Great job!")